# FiftyOne – DDR2019 dataset

Load the processed DDR2019 dataset into FiftyOne for visualization and curation.

**Prerequisites:**
- `data/processed/ddr2019/images/` and `data/processed/ddr2019/labels.csv` (run `uv run preprocess-dataset ddr2019` if missing)
- `FIFTYONE_DATABASE_URI` in `.env` (MongoDB Atlas or local). Same `.env` used by `docker compose -f docker/docker-compose.yml`.
- **MongoDB 6.0+** recommended. If using MongoDB &lt; 6.0, set `FIFTYONE_DATABASE_VALIDATION=false` in `.env` (see .env.sample).

**Paths:** This notebook uses **host paths** (`data/processed/ddr2019/...`) so `fo.launch_app()` on the host can show images. The **Docker FiftyOne App** (port 5151) cannot resolve these host paths; to see images there, run ingestion inside a container with the project mounted (e.g. `/workspace`) and use paths like `/workspace/data/processed/ddr2019/images/...`.

**Troubleshooting (GraphQL / `OperationFailure` when opening the App):** If you see a GraphQL or pymongo `OperationFailure` when loading samples, the DB or datasets may need a migration. With `FIFTYONE_DATABASE_URI` set, run: `fiftyone migrate --info` then `fiftyone migrate --all`.

In [1]:
# Project root and env (run first)
import os
from pathlib import Path

_path = Path.cwd()
while _path != _path.parent and not (_path / "pyproject.toml").exists():
    _path = _path.parent
PROJECT_ROOT = _path if (_path / "pyproject.toml").exists() else Path.cwd()
os.chdir(PROJECT_ROOT)

# Load FIFTYONE_DATABASE_URI from .env (must be before import fiftyone)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
uri = os.environ.get("FIFTYONE_DATABASE_URI")
if not uri:
    raise RuntimeError(
        "FIFTYONE_DATABASE_URI not set. Add it to .env at the project root (see .env.sample)."
    )

import fiftyone as fo
# Use the same MongoDB as .env (recommended: avoids mismatch between notebook and App)
fo.config.database_uri = uri
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/anaya/Develop/Robomous/sam-ai


## Paths and config

- `DATA_DIR`: `data/processed/ddr2019`
- `IMAGES_DIR`: `data/processed/ddr2019/images`
- `LABELS_CSV`: `data/processed/ddr2019/labels.csv` (columns `filename`, `label` 0–4)

Use a **subset** (e.g. 500) for a fast run; set `SUBSET = None` to load all.

In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "ddr2019"
IMAGES_DIR = DATA_DIR / "images"
LABELS_CSV = DATA_DIR / "labels.csv"

SUBSET = 500  # None to load all

if not LABELS_CSV.exists():
    raise FileNotFoundError(f"{LABELS_CSV} not found. Run: uv run preprocess-dataset ddr2019")
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"{IMAGES_DIR} not found. Run: uv run preprocess-dataset ddr2019")

## Ingest into FiftyOne

Create or load dataset `ddr2019`, add samples with `filepath` = host path to image and `ground_truth` = `Classification(label=str(label))`.

In [3]:
import pandas as pd

df = pd.read_csv(LABELS_CSV)
if SUBSET:
    df = df.head(SUBSET)

samples = []
for _, row in df.iterrows():
    fp = IMAGES_DIR / row["filename"]
    if not fp.exists():
        continue
    s = fo.Sample(filepath=str(fp))
    s["ground_truth"] = fo.Classification(label=str(int(row["label"])))
    samples.append(s)

NAME = "ddr2019"
if fo.dataset_exists(NAME):
    ds = fo.load_dataset(NAME)
    ds.delete_samples(ds.values("id"))
    ds.add_samples(samples)
    print(f"Reloaded {NAME} with {len(samples)} samples")
else:
    ds = fo.Dataset(NAME, overwrite=True)
    ds.add_samples(samples)
    ds.persistent = True
    ds.save()
    print(f"Created {NAME} with {len(samples)} samples")

print(ds)

 100% |█████████████████| 500/500 [5.3s elapsed, 0s remaining, 95.0 samples/s]      
Reloaded ddr2019 with 500 samples
Name:        ddr2019
Media type:  image
Num samples: 500
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)


## View in FiftyOne App

- **Option A (Docker):** Set `FIFTYONE_DATASET_NAME=ddr2019` in `.env`, restart the fiftyone service, open http://localhost:5151. The App will not show images when samples use host paths; images need ingestion from inside the container with `/workspace/...` paths.
- **Option B (host, recommended for images):** Run the cell below to launch the App on port **5152** (avoids clash with Docker on 5151) and open in the system browser. Images work because paths are on the host.

In [4]:
# Option B: launch App from notebook (host) on 5152; images will display.
session = fo.launch_app(ds, auto=False, port=5152)
session.open_tab()
# session.wait()  # optional: block until the App tab is closed

Connected to FiftyOne on port 5152 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
Session launched. Run `session.show()` to open the App in a cell output.


<IPython.core.display.Javascript object>